**Note: this notebook requires the "tiler" optional dependency.** It will not work in JupyterLite.

# STAC + Xarray + NDWI + Titiler -> island detection in the Gulf of Morbihan 

This Notebook showcases using [jupyter-tiler](https://github.com/geojupyter/jupyter-tiler) to connect to a STAC catalog and write an island detection algorithm then visualize the result in the JupyterGIS map.

The tiles are computed on demand, so panning and zooming the map triggers computing new areas of the map.

In [ ]:
from jupytergis import GISDocument
from IPython.display import display

latitude = 47.52
longitude = -2.95

doc = GISDocument(latitude=latitude, longitude=longitude, zoom=12)
doc

In [ ]:
doc.add_raster_layer('https://gibs.earthdata.nasa.gov/wmts/epsg3857/best/ASTER_GDEM_Greyscale_Shaded_Relief/default/GoogleMapsCompatible_Level12/{z}/{y}/{x}.jpg')

In [ ]:
from rio_tiler.models import ImageData
import numpy as np


def ndwi_process(data):
    h = data.sizes.get("y", 256)
    w = data.sizes.get("x", 256)

    if "time" in data.dims:
        # pick one scene OR use median over time; choose one
        data = data.isel(time=0)
        # data = data.median(dim="time", skipna=True)

    green = data.sel(band="green").data.astype(np.float32)
    nir = data.sel(band="nir").data.astype(np.float32)

    ndwi = (green - nir) / (green + nir + 1e-6)
    ndwi = np.asarray(ndwi)  # drop masked-array dimensional surprises
    ndwi = np.nan_to_num(ndwi, nan=-1.0, posinf=1.0, neginf=-1.0)

    ndwi_01 = np.clip((ndwi + 1.0) / 2.0, 0.0, 1.0)
    pixels = (ndwi_01 * 255).astype(np.uint8) 

    return ImageData(pixels[np.newaxis, :, :])


await doc.add_stac_array_layer(
    stac_url="https://earth-search.aws.element84.com/v1",
    array_to_image=ndwi_process,
    collection_id="sentinel-2-l2a",
    assets=["green", "nir"],
    datetime="2024-06-01/2024-09-30",
    query={"eo:cloud_cover": {"lt": 5}},
    max_items=3,
    resolution_scale=5,
    resampling="nearest",
)